In [0]:
-- Gold occupation dimension
-- Grain: one row per standardized six-digit SOC code.

CREATE OR REPLACE VIEW workforce_analytics.gold.dim_occupation
COMMENT 'Canonical occupation dimension at the base SOC-code grain, combining BLS and O*NET occupation references.'
AS

WITH occupation_sources AS (

    SELECT
        soc_code,
        occupation_title,
        'bls_employment_projections' AS source_name
    FROM workforce_analytics.silver.bls_employment_projections
    WHERE soc_code RLIKE '^[0-9]{2}-[0-9]{4}$'

    UNION ALL

    SELECT
        soc_code,
        occupation_name AS occupation_title,
        'bls_oews_occupations' AS source_name
    FROM workforce_analytics.silver.bls_oews_occupations
    WHERE soc_code RLIKE '^[0-9]{2}-[0-9]{4}$'

    UNION ALL

    SELECT
        soc_code,
        occupation_title,
        'onet_occupations' AS source_name
    FROM workforce_analytics.silver.onet_occupations
    WHERE soc_code RLIKE '^[0-9]{2}-[0-9]{4}$'
),

occupation_summary AS (
    SELECT
        soc_code,

        COALESCE(
            MAX(
                CASE
                    WHEN source_name = 'bls_employment_projections'
                    THEN occupation_title
                END
            ),
            MAX(
                CASE
                    WHEN source_name = 'bls_oews_occupations'
                    THEN occupation_title
                END
            ),
            MAX(
                CASE
                    WHEN source_name = 'onet_occupations'
                    THEN occupation_title
                END
            )
        ) AS occupation_title,

        CASE
            WHEN MAX(
                CASE
                    WHEN source_name = 'bls_employment_projections'
                    THEN 1 ELSE 0
                END
            ) = 1 THEN TRUE
            ELSE FALSE
        END AS has_employment_projection_data,

        CASE
            WHEN MAX(
                CASE
                    WHEN source_name = 'bls_oews_occupations'
                    THEN 1 ELSE 0
                END
            ) = 1 THEN TRUE
            ELSE FALSE
        END AS has_oews_data,

        CASE
            WHEN MAX(
                CASE
                    WHEN source_name = 'onet_occupations'
                    THEN 1 ELSE 0
                END
            ) = 1 THEN TRUE
            ELSE FALSE
        END AS has_onet_data

    FROM occupation_sources
    GROUP BY soc_code
),

onet_counts AS (
    SELECT
        soc_code,
        COUNT(DISTINCT onet_soc_code) AS onet_occupation_count
    FROM workforce_analytics.silver.onet_occupations
    WHERE soc_code RLIKE '^[0-9]{2}-[0-9]{4}$'
    GROUP BY soc_code
)

SELECT
    occupation.soc_code,
    occupation.occupation_title,

    CONCAT(
        SUBSTRING(occupation.soc_code, 1, 2),
        '-0000'
    ) AS major_group_code,

    major_group.occupation_title AS major_group_title,

    COALESCE(
        onet_counts.onet_occupation_count,
        0
    ) AS onet_occupation_count,

    occupation.has_employment_projection_data,
    occupation.has_oews_data,
    occupation.has_onet_data

FROM occupation_summary AS occupation

LEFT JOIN occupation_summary AS major_group
    ON major_group.soc_code = CONCAT(
        SUBSTRING(occupation.soc_code, 1, 2),
        '-0000'
    )

LEFT JOIN onet_counts
    ON occupation.soc_code = onet_counts.soc_code;
